In [ ]:
# from datasets import load_dataset

# ds = load_dataset("stablegradients/maze-17x17-500k-general", "binary")
# # print(ds["train"][0])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls drive/MyDrive  

In [4]:
# ds["train"][0]

In [5]:
# len(ds["train"][99]["extra_info"]["question"].split())

In [6]:
# ds["train"][99]["extra_info"]["question"].split()

In [ ]:
# len(ds["train"][9]["extra_info"]["answer"].split())

In [ ]:
# 310-17*17

In [2]:
import json

with open("data/val.json", "r") as f:
    file = json.loads(f.read())

#

In [3]:
# file

In [ ]:
s = file[0]
s

In [13]:
# enc_ = tokenizer.encode(s.split())

In [15]:
# dec_ = tokenizer.decode(enc_)
# dec_

In [ ]:
%cd /content/drive/MyDrive/anth

In [5]:
from model import NanoQwen
from config import NanoQwenConfig
import torch
from dataset import _tokenizer


In [6]:
config = NanoQwenConfig()
model = NanoQwen(config)
tokenizer = _tokenizer()

# input_ids = torch.randint(0, config.vocab_size, (2, 16))
# targets = torch.randint(0, config.vocab_size, (2, 16))

# logits, loss = model(input_ids, targets)
# print(f"Logits shape: {logits.shape}, Loss: {loss.item()}")

In [ ]:
model.generate(tokenizer, s)

In [ ]:
s = file[0]["sequence"]
s

In [28]:
list_strings = [s,s,s]

In [54]:
# [g.ids for g in tokenizer.encode_batch(list_strings)]

In [8]:
temperature=1
model.eval()


input_ids = tokenizer.encode(s).ids

x = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0)
x = x.to(next(model.parameters()).device)

In [9]:
eos_id = tokenizer.encode("<eos>").ids[0]

In [10]:
import torch.nn.functional as F

generated = []


for _ in range(NanoQwenConfig.max_seq_len - x.size(1)):
    logits, _ = model(x)

    if temperature ==0:
        next_id = logits.argmax(dim=-1, keepdim=True)

    else:
        probs = F.softmax(logits/temperature, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
    

    generated.append(next_id.item())
    if next_id.item() == eos_id: break

    x= torch.cat([x, next_id], dim=-1)

    


In [ ]:
tokenizer.decode(generated)

In [ ]:
model(input_ids)[0].shape

In [ ]:
torch.manual_seed(42)
while x.size(1) < 643:
    with torch.no_grad():
        logits, _ = model(x, x)
        
        
        logits = logits[:, -1, :]

        
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_indices = torch.topk(probs, k=5, dim=-1)

        

        ix = torch.multinomial(topk_probs, num_samples=1)


        xcol= torch.gather(topk_indices, dim=-1, index=ix)

        x = torch.cat((x, xcol), dim=1)
        

In [ ]:
tokenizer.decode(x)

In [ ]:
x.shape

In [ ]:
17*17

In [ ]:
len(s.split())

In [ ]:
310-289

In [ ]:
s.split()[-29:]

In [ ]:
%cd /content/drive/MyDrive/anth

from model import NanoQwen
from config  import RLHFConfig
from dataset import MazeDataset, _tokenizer, collate_fn
from rollout import rollout
tokenizer=_tokenizer()

config = RLHFConfig()
m = NanoQwen(config).to("cuda")

val_dataset = MazeDataset(tokenizer=tokenizer, 
                                json_file_path="data/val.json")


P
prompt = next(iter(val_dataset))[0].to("cuda").view(1, -1)

f = rollout(m, prompt, config)

In [ ]:
tokenizer.encode("<eos>").ids


In [ ]:
prompt[0].shape[0]